# Graphyco Full Tutorial & JSON Export

This notebook provides a complete tutorial on how to use Graphyco with a current training run, how to utilize the JSON export options, and details on when it is valid to use FX tracing.

## When is it valid to trace the network?

Graphyco uses PyTorch FX for its `mode="trace"` by default. This symbolic tracing mechanism maps the operations into a directed graph and is valid only for models that have a **static computational graph**.

**Valid for Tracing:**
- Standard feed-forward networks (MLPs, CNNs, ResNets, standard Transformers).
- Control flow that depends ONLY on static configuration (e.g., `if self.use_dropout:`).

**Invalid for Tracing:**
- Dynamic data-dependent control flow (e.g., `if x.sum() > 0:` where `x` is a tensor).
- Certain third-party C++ bindings that FX cannot interpret.
- Recurrent Neural Networks (RNNs/LSTMs) where the loop depends on sequence length dynamically.

If your model cannot be traced, you should initialize Graphyco utilities with `mode="module"` which uses a fallback recursive `nn.Module` inspector.

## Live Training Monitor

Let's set up a standard PyTorch training loop and attach the `LiveTrainingMonitor` to it to track bottlenecks in real-time.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from graphyco import LiveTrainingMonitor, extract_benchmark_json, QueryEngine
import os

# Define a simple traceable model
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Initialize the monitor
# mode="trace" is valid here because our Sequential model has a static graph.
monitor = LiveTrainingMonitor(model, mode="trace", log_interval=5)

print("Starting training loop...")
for step in range(20):
    x = torch.randn(16, 64)
    y = torch.randint(0, 10, (16,))

    # Wrap the forward, loss, backward, and optimizer steps in the observe context
    with monitor.observe(step):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    # Retrieve real-time metrics
    if monitor.has_new_data():
        bneck = monitor.get_latest_bottleneck()
        print(f"Step {step}: Choke Node = {bneck.get('bottleneck_node')} (G_max={bneck.get('max_concentration', 0):.2f})")

print("Training complete.")

## JSON Export for Offline Analysis

To export the model's structural and dynamic gradient flow data to a JSON format that can be used by the `QueryEngine` or the visualizer desktop app, we use `extract_benchmark_json`.

In [ ]:
# We run a short automated profiling loop over our model to export the full benchmark JSON.
x_sample = torch.randn(16, 64)
json_path = "my_model_benchmark.json"

# This automatically runs forward/backward passes and exports the JSON
extract_benchmark_json(
    target=model, 
    inputs=x_sample, 
    steps=5, 
    arch_name="Tutorial_Model",
    export_path=json_path
)

print(f"Exported comprehensive benchmark to {json_path}")
print(f"File size: {os.path.getsize(json_path)} bytes")

## Using the Query Engine on Exported JSON

You can use the `QueryEngine` to inspect the exported JSON offline, enabling automated analysis without needing the original model loaded in memory.

In [ ]:
# Load the exported JSON
engine = QueryEngine(json_path)

# Describe the summary
summary_df = engine.describe()
display(summary_df)

# Retrieve bottlenecks
bottlenecks = engine.get_bottlenecks()
for b in bottlenecks:
    print(f"{b.get('architecture')}: G_max={b.get('gradient_bottleneck_gmax')} at node '{b.get('bottleneck_node')}'")